#### 1. SETUP & CONFIGURATION

In [3]:
import sys
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, lit, explode, when, input_file_name, regexp_extract, 
    monotonically_increasing_id, udf, array, when
)
from pyspark.sql.types import StringType, IntegerType, StructType, StructField

# 1. INITIALIZE CONTEXT
#sc = SparkContext()
#glueContext = GlueContext(sc)
#spark = glueContext.spark_session

# --- CONFIGURATION ---
S3_BASE_PATH = "s3://project-matches-bucket/matches-raw-data" 
OUTPUT_PATH = "s3://project-matches-bucket/warehouse/production"

#  PRODUCTION MODE: Process ALL files 
TEST_MODE = False 

print(f"🚀 Starting Full Production ETL.")
print(f"📂 Input: {S3_BASE_PATH}")
print(f"📂 Output: {OUTPUT_PATH}")

🚀 Starting Full Production ETL.
📂 Input: s3://project-matches-bucket/matches-raw-data
📂 Output: s3://project-matches-bucket/warehouse/production


#### DIMENSION: COMPETITION & SEASON
Source: competitions.json

In [9]:
print("Processing Competitions & Seasons...")

comp_df = spark.read.option("multiline", "true").json(f"{S3_BASE_PATH}/competitions.json")

# Dim_Competition
dim_competition = comp_df.select(
    col("competition_id"),
    col("competition_name"),
    col("country_name").alias("country"),
    col("competition_gender"),
    col("competition_youth"),
    col("competition_international")
).dropDuplicates(["competition_id"])

# Dim_Season
dim_season = comp_df.select(
    col("season_id"),
    col("season_name"),
    col("competition_id")
).dropDuplicates(["season_id"])

# Write
dim_competition.write.mode("overwrite").parquet(f"{OUTPUT_PATH}/dim_competition/")
dim_season.write.mode("overwrite").parquet(f"{OUTPUT_PATH}/dim_season/")
print("✅ Competitions & Seasons Saved.")

Processing Competitions & Seasons...
Dim_Competition Preview:
+--------------+----------------+-------+------------------+-----------------+-------------------------+
|competition_id|competition_name|country|competition_gender|competition_youth|competition_international|
+--------------+----------------+-------+------------------+-----------------+-------------------------+
|             2|  Premier League|England|              male|            false|                    false|
|             7|         Ligue 1| France|              male|            false|                    false|
|             9|   1. Bundesliga|Germany|              male|            false|                    false|
|            11|         La Liga|  Spain|              male|            false|                    false|
|            12|         Serie A|  Italy|              male|            false|                    false|
+--------------+----------------+-------+------------------+-----------------+--------------------

#### 3. DIMENSION: MATCH, MANAGER, REFEREE, STADIUM, TEAM
Source: matches/ folder (Nested JSON files)

In [8]:
print("Processing Matches & Context Dimensions...")

# Path selection
matches_path = f"{S3_BASE_PATH}/matches/*/*.json"
if TEST_MODE: matches_path = f"{S3_BASE_PATH}/matches/11/4.json"

raw_matches_df = spark.read.option("multiline", "true").json(matches_path)

# --- Dim_Manager ---
# FIX: We extract the country from the MANAGER object, not the team object.
home_mgr = raw_matches_df.select(explode(col("home_team.managers")).alias("mgr")).select(col("mgr"), col("mgr.country.name").alias("manager_country"))
away_mgr = raw_matches_df.select(explode(col("away_team.managers")).alias("mgr")).select(col("mgr"), col("mgr.country.name").alias("manager_country"))

all_managers = home_mgr.union(away_mgr).select(
    col("mgr.id").alias("manager_id"),
    col("mgr.name").alias("manager_name"),
    col("mgr.nickname").alias("manager_nickname"),
    col("mgr.dob").alias("manager_dob"),
    col("manager_country")
).dropDuplicates(["manager_id"])

# --- Dim_Referee ---
if "referee" in raw_matches_df.columns:
    dim_referee = raw_matches_df.select(
        col("referee.id").alias("referee_id"),
        col("referee.name").alias("referee_name"),
        col("referee.country.name").alias("country")
    ).dropna(subset=["referee_id"]).dropDuplicates(["referee_id"])
else:
    # Handle case where no referee data exists
    schema = StructType([StructField("referee_id", IntegerType()), StructField("referee_name", StringType()), StructField("country", StringType())])
    dim_referee = spark.createDataFrame([], schema)

# --- Dim_Stadium ---
if "stadium" in raw_matches_df.columns:
    dim_stadium = raw_matches_df.select(
        col("stadium.id").alias("stadium_id"),
        col("stadium.name").alias("stadium_name"),
        col("stadium.country.name").alias("country"),
        lit(None).cast(StringType()).alias("city") 
    ).dropna(subset=["stadium_id"]).dropDuplicates(["stadium_id"])
else:
    schema = StructType([StructField("stadium_id", IntegerType()), StructField("stadium_name", StringType()), StructField("city", StringType()), StructField("country", StringType())])
    dim_stadium = spark.createDataFrame([], schema)

# --- Dim_Team ---
home_teams = raw_matches_df.select(
    col("home_team.home_team_id").alias("team_id"),
    col("home_team.home_team_name").alias("team_name"),
    col("home_team.country.name").alias("country"),
    col("home_team.home_team_gender").alias("team_gender"),
    lit("Club").alias("type") 
)
away_teams = raw_matches_df.select(
    col("away_team.away_team_id").alias("team_id"),
    col("away_team.away_team_name").alias("team_name"),
    col("away_team.country.name").alias("country"),
    col("away_team.away_team_gender").alias("team_gender"),
    lit("Club").alias("type")
)
dim_team = home_teams.union(away_teams).dropDuplicates(["team_id"])

# --- Dim_Match ---
dim_match = raw_matches_df.select(
    col("match_id"),
    col("competition.competition_id").alias("competition_id"),
    col("season.season_id").alias("season_id"),
    col("match_date"),
    col("kick_off"),
    col("home_team.home_team_id").alias("home_team_id"),
    col("away_team.away_team_id").alias("away_team_id"),
    col("home_team.managers")[0]["id"].alias("home_manager_id"),
    col("away_team.managers")[0]["id"].alias("away_manager_id"),
    col("stadium.id").alias("stadium_id"),
    col("referee.id").alias("referee_id"),
    col("home_score"),
    col("away_score"),
    col("competition_stage.name").alias("competition_stage")
).dropDuplicates(["match_id"])

# Write
all_managers.write.mode("overwrite").parquet(f"{OUTPUT_PATH}/dim_manager/")
dim_referee.write.mode("overwrite").parquet(f"{OUTPUT_PATH}/dim_referee/")
dim_stadium.write.mode("overwrite").parquet(f"{OUTPUT_PATH}/dim_stadium/")
dim_team.write.mode("overwrite").parquet(f"{OUTPUT_PATH}/dim_team/")
dim_match.write.mode("overwrite").parquet(f"{OUTPUT_PATH}/dim_match/")
print("✅ Matches & Context Dimensions Saved.")

Processing Matches & Context Dimensions...
Dim_Match Preview:
+--------+--------------+---------+----------+------------+------------+------------+---------------+---------------+----------+----------+----------+----------+-----------------+
|match_id|competition_id|season_id|match_date|    kick_off|home_team_id|away_team_id|home_manager_id|away_manager_id|stadium_id|referee_id|home_score|away_score|competition_stage|
+--------+--------------+---------+----------+------------+------------+------------+---------------+---------------+----------+----------+----------+----------+-----------------+
|   15946|            11|        4|2018-08-18|22:15:00.000|         217|         206|            227|            187|       342|      2625|         3|         0|   Regular Season|
|   15956|            11|        4|2018-08-25|22:15:00.000|         901|         217|            576|            227|       622|      2602|         0|         1|   Regular Season|
|   15973|            11|        4|201

#### 4. DIMENSION: PLAYER, POSITION & FACT LINEUPS (Bridge)
Source: lineups/ folder

In [25]:
print("Processing Lineups...")

lineups_path = f"{S3_BASE_PATH}/lineups/*.json"
if TEST_MODE: lineups_path = f"{S3_BASE_PATH}/lineups/15946.json"

raw_lineups_df = spark.read.option("multiline", "true").json(lineups_path)

# Explode Logic
teams_in_lineup = raw_lineups_df.select(col("team_id"), col("lineup"))

fact_lineups = teams_in_lineup.withColumn("filename", input_file_name()) \
    .withColumn("match_id", regexp_extract("filename", r"(\d+)\.json", 1).cast(IntegerType())) \
    .select(col("match_id"), col("team_id"), explode("lineup").alias("player")) \
    .select(
        col("match_id"),
        col("team_id"),
        col("player.player_id").alias("player_id"),
        col("player.jersey_number").alias("shirt_number"),
        col("player.positions")[0]["position_id"].alias("position_id"),
        col("player.positions")[0]["position"].alias("position_name"),
        col("player.positions")[0]["position_id"].isNotNull().alias("is_starter")
    )

# Dim_Player
dim_player = raw_lineups_df.select(explode("lineup").alias("player")).select(
    col("player.player_id").alias("player_id"),
    col("player.player_name").alias("player_name"),
    col("player.player_nickname").alias("nickname"),
    col("player.country.name").alias("country"),
    lit(None).cast(StringType()).alias("birth_date"),
    lit(None).cast(StringType()).alias("role"),
    lit(None).cast(StringType()).alias("footedness"),
    lit(None).cast(IntegerType()).alias("height"),
    lit(None).cast(IntegerType()).alias("weight")
).dropDuplicates(["player_id"])

# Dim_Position (With Group Logic)
dim_position = fact_lineups.select(
    col("position_id"),
    col("position_name")
).dropna(subset=["position_id"]).dropDuplicates(["position_id"]).withColumn(
    "position_group",
    when(col("position_id") == 1, "Goalkeeper")
    .when(col("position_id").isin([2, 3, 4, 5, 6, 7, 8]), "Defender")
    .when(col("position_id").isin([9, 10, 11, 12, 13, 14, 15, 16, 18, 19, 20]), "Midfielder")
    .when(col("position_id").isin([17, 21, 22, 23, 24, 25]), "Forward")
    .otherwise("Unknown") 
)

# Write
dim_player.write.mode("overwrite").parquet(f"{OUTPUT_PATH}/dim_player/")
dim_position.write.mode("overwrite").parquet(f"{OUTPUT_PATH}/dim_position/")
fact_lineups.write.mode("overwrite").parquet(f"{OUTPUT_PATH}/fact_lineups/")
print("✅ Lineups & Players Saved.")

Processing Lineups (Players & Positions)...
Fact Lineups Preview:
+--------+-------+---------+------------+-----------+--------------------+----------+
|match_id|team_id|player_id|shirt_number|position_id|       position_name|is_starter|
+--------+-------+---------+------------+-----------+--------------------+----------+
|   15946|    217|     3109|          14|       null|                null|     false|
|   15946|    217|     3501|           7|         15|Left Center Midfield|      true|
|   15946|    217|     5203|           5|          9|Right Defensive M...|      true|
|   15946|    217|     5211|          18|          6|           Left Back|      true|
|   15946|    217|     5213|           3|          3|   Right Center Back|      true|
+--------+-------+---------+------------+-----------+--------------------+----------+
only showing top 5 rows

dim_player Preview:
+---------+--------------------+-----------------+-------+----------+----+----------+------+------+
|player_id|    

#### 5. DIMENSION: EVENT TYPE (Scanning a sample)
Source: events/ folder

In [34]:
print("Processing Event Types...")

events_path = f"{S3_BASE_PATH}/events/*.json"
if TEST_MODE: events_path = f"{S3_BASE_PATH}/events/15946.json"

raw_events_df = spark.read.option("multiline", "true").json(events_path)
raw_events_df.createOrReplaceTempView("events")

# Expanded SQL to catch more subtypes
dim_event_type_sql = """
SELECT DISTINCT
    e.type.id as statsbomb_type_id,
    e.type.name as event_type_name,
    COALESCE(
        e.pass.type.id, 
        e.shot.type.id, 
        e.duel.type.id, 
        e.foul_committed.type.id,
        e.goalkeeper.type.id,
        e.dribble.outcome.id,
        e.interception.outcome.id
    ) as statsbomb_subtype_id,
    COALESCE(
        e.pass.type.name, 
        e.shot.type.name, 
        e.duel.type.name, 
        e.foul_committed.type.name,
        e.goalkeeper.type.name,
        e.dribble.outcome.name,
        e.interception.outcome.name
    ) as event_subtype_name,
    'Action' as event_category
FROM events e
"""
dim_event_type = spark.sql(dim_event_type_sql)
dim_event_type = dim_event_type.withColumn("event_type_sk", monotonically_increasing_id())

# Cache for join efficiency
dim_event_type.cache()
dim_event_type.write.mode("overwrite").parquet(f"{OUTPUT_PATH}/dim_event_type/")
print("✅ Event Types Saved.")

Processing Event Types...
Dim_EventType Preview:
+-----------------+---------------+--------------------+------------------+--------------+-------------+
|statsbomb_type_id|event_type_name|statsbomb_subtype_id|event_subtype_name|event_category|event_type_sk|
+-----------------+---------------+--------------------+------------------+--------------+-------------+
|               14|        Dribble|                   8|          Complete|        Action|            0|
|               23|    Goal Keeper|                  33|        Shot Saved|        Action|            1|
|               30|           Pass|                  63|         Goal Kick|        Action|            2|
|               19|   Substitution|                null|              null|        Action|            3|
|                6|          Block|                null|              null|        Action|            4|
|               30|           Pass|                  66|          Recovery|        Action|            5|
|     

#### 6. FACT TABLE: FACT EVENTS
This processes the main event stream, flattens it, and links to dimensions.


In [35]:
print("Processing Fact Events...")

# 1. Add Match ID
fact_events_base = raw_events_df.withColumn("filename", input_file_name()) \
    .withColumn("match_id", regexp_extract("filename", r"(\d+)\.json", 1).cast(IntegerType()))

# 2. Define Join Key Logic
def get_subtype_id(p, s, d, f, g, dr, i):
    return next((x for x in [p, s, d, f, g, dr, i] if x is not None), None)

subtype_udf = udf(get_subtype_id, IntegerType())

# 3. Flatten
flattened_events = fact_events_base.select(
    col("id").alias("event_id"),
    col("match_id"),
    col("team.id").alias("team_id"),
    col("player.id").alias("player_id"),
    col("possession").alias("possession_id"),
    col("possession_team.id").alias("possession_team_id"),
    col("period"),
    col("minute"),
    col("second"),
    col("timestamp"),
    col("play_pattern.id").alias("play_pattern_id"),
    col("duration"),
    
    # Join Keys
    col("type.id").alias("join_type_id"),
    subtype_udf(
        col("pass.type.id"), 
        col("shot.type.id"), 
        col("duel.type.id"), 
        col("foul_committed.type.id"),
        col("goalkeeper.type.id"),
        col("dribble.outcome.id"),
        col("interception.outcome.id")
    ).alias("join_subtype_id"),

    # Locations
    col("location")[0].alias("location_x"),
    col("location")[1].alias("location_y"),
    
    # End Locations
    when(col("pass").isNotNull(), col("pass.end_location")[0])
      .when(col("carry").isNotNull(), col("carry.end_location")[0])
      .when(col("shot").isNotNull(), col("shot.end_location")[0])
      .alias("end_location_x"),
      
    when(col("pass").isNotNull(), col("pass.end_location")[1])
      .when(col("carry").isNotNull(), col("carry.end_location")[1])
      .when(col("shot").isNotNull(), col("shot.end_location")[1])
      .alias("end_location_y"),

    # Event Specifics
    col("pass.length").alias("pass_length"),
    col("pass.angle").alias("pass_angle"),
    col("pass.height.name").alias("pass_height"), 
    col("pass.outcome.name").alias("pass_outcome"),
    col("pass.recipient.id").alias("pass_recipient_id"),

    col("carry.end_location")[0].alias("carry_end_x"),
    col("carry.end_location")[1].alias("carry_end_y"),
    # col("carry.length") removed as it doesn't exist

    col("shot.statsbomb_xg").alias("shot_xg"),
    col("shot.outcome.name").alias("shot_outcome"),
    col("shot.technique.name").alias("shot_technique"),
    col("shot.body_part.name").alias("shot_body_part"),
    col("shot.end_location")[0].alias("shot_end_location_x"),
    col("shot.end_location")[1].alias("shot_end_location_y"),

    col("dribble.outcome.name").alias("dribble_outcome"),

    when(col("foul_committed").isNotNull(), col("foul_committed.type.name"))
      .when(col("foul_won").isNotNull(), "Foul Won") 
      .alias("foul_type"),
      
    col("block.deflection").alias("block_deflection"),
    col("clearance.body_part.name").alias("clearance_body_part"),
    col("interception.outcome.name").alias("interception_outcome"),

    col("under_pressure"),
    col("counterpress"),
    col("off_camera"),
    col("related_events").alias("related_event_ids")
)

# 4. Join to get Surrogate Key
final_fact_events = flattened_events.join(
    dim_event_type,
    (flattened_events.join_type_id == dim_event_type.statsbomb_type_id) & 
    (flattened_events.join_subtype_id.eqNullSafe(dim_event_type.statsbomb_subtype_id)),
    "left"
).select(
    "event_id", "match_id", "team_id", "player_id", 
    "possession_id", "possession_team_id", 
    "period", "minute", "second", "timestamp", 
    "play_pattern_id", "duration",
    col("event_type_sk"), 
    "location_x", "location_y", "end_location_x", "end_location_y",
    "pass_length", "pass_angle", "pass_height", "pass_outcome", "pass_recipient_id",
    "carry_end_x", "carry_end_y", 
    "shot_xg", "shot_outcome", "shot_technique", "shot_body_part", "shot_end_location_x", "shot_end_location_y",
    "dribble_outcome", "foul_type", "block_deflection", "clearance_body_part", "interception_outcome",
    "under_pressure", "counterpress", "off_camera", "related_event_ids"
)

# Write
final_fact_events.write.mode("overwrite").parquet(f"{OUTPUT_PATH}/fact_events/")
print("✅ All Facts Saved. ETL Complete!")

Processing Fact Events (This may take time)...
Fact Events Preview:
+--------------------+--------+-------+---------+-------------+------------------+------+------+------+------------+---------------+--------+-------------+----------+----------+--------------+--------------+-----------+----------+-----------+------------+-----------------+-----------+-----------+-------+------------+--------------+--------------+-------------------+-------------------+---------------+---------+----------------+-------------------+--------------------+--------------+------------+----------+--------------------+
|            event_id|match_id|team_id|player_id|possession_id|possession_team_id|period|minute|second|   timestamp|play_pattern_id|duration|event_type_sk|location_x|location_y|end_location_x|end_location_y|pass_length|pass_angle|pass_height|pass_outcome|pass_recipient_id|carry_end_x|carry_end_y|shot_xg|shot_outcome|shot_technique|shot_body_part|shot_end_location_x|shot_end_location_y|dribble_out